<div style="background:linear-gradient(135deg,#0a2540 0%,#1a3a5c 60%,#0f3460 100%);
            padding:40px 30px;border-radius:12px;margin-bottom:20px">
  <h1 style="color:#ffffff;font-size:2.2em;margin:0 0 8px 0">
    🧬 Clasificación QSAR basada en Docking: Activos vs Decoys
  </h1>
  <p style="color:#a8c4e0;font-size:1.1em;margin:0">
    NB-ML-03 · Fingerprints ProLIF · XGBoost · Dark Chemical Matter · UNAL 2026
  </p>
</div>


---
## Objetivo del notebook

Construimos un clasificador que aprende a distinguir **activos reales** (inhibidores
de ChEMBL) de **decoys** (moléculas con propiedades similares pero scaffolds distintos)
a partir de sus **fingerprints de interacción ProLIF** en el sitio de unión.

### Pipeline completo

```
molecules_prepared.csv (activos ChEMBL con PDBQT)
    ↓ Scaffolds genéricos (MakeScaffoldGeneric)
    ↓ Selección de decoys: Dark Chemical Matter (DCM) con propiedades similares, scaffolds distintos
    ↓ Docking Vina (mismos parámetros que NB-DOCK-02)
    ↓ ProLIF fingerprints (activos + decoys)
    ↓ DataFrame combinado con etiqueta activity (1=activo, 0=decoy)
    ↓ 3 splits: random · scaffold · fingerprint
    ↓ XGBoost (F1, MCC, Balanced Accuracy)
    ↓ Predicción sobre Dark Chemical Matter (DCM) (30 moléculas)
```


---
## 1. Instalación y configuración

In [ ]:
# ── Instalar librerías ───────────────────────────────────────────────────────
!pip install rdkit prolif meeko vina xgboost deepchem scikit-learn pandas numpy matplotlib seaborn tqdm joblib gemmi --quiet
print("✅ Librerías instaladas")


In [1]:
# ── Importaciones ────────────────────────────────────────────────────────────
import os, json, warnings, requests, io
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

from tqdm.auto import tqdm
from rdkit import Chem
from rdkit.Chem import Descriptors, AllChem, Draw
from rdkit.Chem.Scaffolds import MurckoScaffold
import prolif as plf
from meeko import MoleculePreparation
from vina import Vina

try:
    import xgboost as xgb
    XGB_OK = True
    print(f"✅ XGBoost {xgb.__version__}")
except ImportError:
    XGB_OK = False

print("✅ Importaciones completadas")


/home/dfvic/.local/lib/python3.14/site-packages/MDAnalysis/topology/tables.py:52: DeprecationWarning: Deprecated in version 2.8.0
MDAnalysis.topology.tables has been moved to MDAnalysis.guesser.tables. This import point will be removed in MDAnalysis version 3.0.0
  warnings.warn(wmsg, category=DeprecationWarning)


✅ XGBoost 3.2.0
✅ Importaciones completadas


In [2]:
# ── Parámetros del sistema + descarga de archivos necesarios ─────────────────
import requests, os, json
import numpy as np

PDB_ID        = '7WJO'
LIGAND_CODE   = 'BGI'
TARGET_NAME   = 'TRPV1'

protein_directory = 'estructuras'
pdbqt_directory   = 'pdbqt'
docks_directory   = 'docking'
decoys_directory  = 'docking_decoys'

for d in [protein_directory, pdbqt_directory, docks_directory,
          decoys_directory, 'mols', 'results']:
    os.makedirs(d, exist_ok=True)

BASE = "https://raw.githubusercontent.com/FelPVic/curso_datascience/main/files"

# Todos los SDFs de activos disponibles en el repo
CHEMBL_IDS = [
    'CHEMBL1791178','CHEMBL1791436','CHEMBL265094','CHEMBL285056',
    'CHEMBL285445','CHEMBL287110','CHEMBL287375','CHEMBL32125',
    'CHEMBL32495','CHEMBL32687','CHEMBL32780','CHEMBL34818',
    'CHEMBL35306','CHEMBL35365','CHEMBL35516','CHEMBL35671',
    'CHEMBL35735','CHEMBL35760','CHEMBL35895','CHEMBL36240',
    'CHEMBL36409','CHEMBL36445','CHEMBL36446','CHEMBL412808',
    'CHEMBL416596','CHEMBL422402','CHEMBL443517','CHEMBL4741395',
    'CHEMBL4786444',
]

archivos_necesarios = {
    # estructuras
    f'{protein_directory}/7WJO_a.pdb':         f'{BASE}/estructuras/7WJO_a.pdb',
    f'{protein_directory}/7WJO_a_clean.pdb':   f'{BASE}/estructuras/7WJO_a_clean.pdb',
    f'{protein_directory}/BGI_org.sdf':         f'{BASE}/estructuras/BGI_org.sdf',
    f'{protein_directory}/BGI_org.pdb':         f'{BASE}/estructuras/BGI_org.pdb',
    f'{protein_directory}/docking_params.json': f'{BASE}/estructuras/docking_params.json',
    f'{protein_directory}/protein_h.pdb':       f'{BASE}/estructuras/protein_h.pdb',
    # pdbqt
    f'{pdbqt_directory}/7WJO.pdbqt':           f'{BASE}/pdbqt/7WJO.pdbqt',
    f'{pdbqt_directory}/BGI.pdbqt':            f'{BASE}/pdbqt/BGI.pdbqt',
    # SDFs de activos
    **{f'{docks_directory}/{c}.sdf': f'{BASE}/docking/{c}.sdf' for c in CHEMBL_IDS},
    # CSVs
    'fps_activos.csv':        f'{BASE}/docking/fps_activos.csv',
    'molecules_prepared.csv': f'{BASE}/mols/molecules_prepared.csv',
}

print("Verificando y descargando archivos del repositorio...")
print("=" * 60)
descargados, existentes, fallidos = 0, 0, 0

for destino, url in archivos_necesarios.items():
    if os.path.exists(destino):
        existentes += 1
        continue
    nombre = os.path.basename(destino)
    try:
        r = requests.get(url, timeout=120)
        if r.status_code == 200:
            with open(destino, 'wb') as fh:
                fh.write(r.content)
            print(f"  📥 {nombre:<45} ({len(r.content)/1024:.1f} KB)")
            descargados += 1
        else:
            print(f"  ❌ {nombre:<45} HTTP {r.status_code}")
            fallidos += 1
    except Exception as e:
        print(f"  ❌ {nombre:<45} {e}")
        fallidos += 1

print()
print(f"  ✅ Ya existían: {existentes}  |  📥 Descargados: {descargados}  |  ❌ Fallidos: {fallidos}")

# Cargar parámetros de docking
params_path = f'{protein_directory}/docking_params.json'
if os.path.exists(params_path):
    with open(params_path) as fh:
        dp = json.load(fh)
    pocket_center = np.array(dp['pocket_center'])
    ligand_box    = np.array(dp['ligand_box'])
    print(f"\n✅ Parámetros de docking cargados:")
    print(f"   Centro: {pocket_center}")
    print(f"   Caja:   {ligand_box}")
else:
    print("\n⚠️  docking_params.json no encontrado — define manualmente:")
    pocket_center = np.array([0.0, 0.0, 0.0])
    ligand_box    = np.array([20.0, 20.0, 20.0])


Verificando y descargando archivos del repositorio...
  📥 7WJO_a.pdb                                    (523.5 KB)
  📥 7WJO_a_clean.pdb                              (508.7 KB)
  📥 BGI_org.sdf                                   (3.0 KB)
  📥 BGI_org.pdb                                   (18.3 KB)
  📥 docking_params.json                           (0.3 KB)
  📥 protein_h.pdb                                 (1011.5 KB)
  📥 7WJO.pdbqt                                    (610.5 KB)
  📥 BGI.pdbqt                                     (4.8 KB)
  📥 CHEMBL1791178.sdf                             (20.2 KB)
  📥 CHEMBL1791436.sdf                             (22.7 KB)
  📥 CHEMBL265094.sdf                              (10.6 KB)
  📥 CHEMBL285056.sdf                              (12.0 KB)
  📥 CHEMBL285445.sdf                              (10.9 KB)
  📥 CHEMBL287110.sdf                              (13.4 KB)
  📥 CHEMBL287375.sdf                              (9.1 KB)
  📥 CHEMBL32125.sdf                          

---
## 2. Scaffolds genéricos de los activos ChEMBL

Calculamos el **scaffold genérico** (MakeScaffoldGeneric) de cada activo —
elimina todos los heteroátomos y sustituye con carbonos, dejando solo la topología
del esqueleto. Esto permite una comparación más robusta que el scaffold de Murcko
para identificar decoys estructuralmente distintos.


In [3]:
# ── Cargar activos preparados ────────────────────────────────────────────────
# Desde archivo local o desde GitHub
ARCHIVO_MOL = 'molecules_prepared.csv'
URL_MOL = ("https://raw.githubusercontent.com/FelPVic/curso_datascience/"
           "main/files/molecules_prepared.csv")

if os.path.exists(ARCHIVO_MOL):
    df_activos = pd.read_csv(ARCHIVO_MOL)
    print(f"✅ Activos cargados localmente: {len(df_activos)}")
else:
    print("Descargando molecules_prepared.csv desde GitHub...")
    df_activos = pd.read_csv(URL_MOL)
    print(f"✅ Activos cargados desde GitHub: {len(df_activos)}")

# Normalizar columna de pActividad
if 'pValue' in df_activos.columns and 'pActividad' not in df_activos.columns:
    df_activos = df_activos.rename(columns={'pValue': 'pActividad'})

df_activos = df_activos.dropna(subset=['std_smiles']).reset_index(drop=True)
print(f"   Columnas: {df_activos.columns.tolist()}")


✅ Activos cargados localmente: 29
   Columnas: ['canonical_smiles', 'molecule_chembl_id', 'pchembl_value', 'standard_type', 'standard_units', 'standard_value', 'type', 'units', 'value', 'pActividad', 'mol_weight', 'std_smiles', 'cur_error', 'pdbqt']


In [4]:
# ── Calcular scaffolds genéricos de los activos ──────────────────────────────
def scaffold_generico(smiles):
    """
    Calcula el scaffold genérico (MakeScaffoldGeneric):
    - Extrae el scaffold de Murcko (elimina cadenas laterales)
    - Reemplaza todos los heteroátomos por C (genérico)
    - Convierte todos los enlaces a simples (topología pura)
    Retorna el SMILES del scaffold o None si falla.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    try:
        # Paso 1: Murcko scaffold
        core = MurckoScaffold.GetScaffoldForMol(mol)
        # Paso 2: scaffold genérico
        generic = MurckoScaffold.MakeScaffoldGeneric(core)
        return Chem.MolToSmiles(generic)
    except Exception:
        return None

df_activos['scaffold_murcko'] = df_activos['std_smiles'].apply(
    lambda s: Chem.MolToSmiles(MurckoScaffold.GetScaffoldForMol(
        Chem.MolFromSmiles(s))) if Chem.MolFromSmiles(s) else None
)
df_activos['scaffold_generico'] = df_activos['std_smiles'].apply(scaffold_generico)

scaffolds_activos = set(df_activos['scaffold_generico'].dropna().tolist())

print(f"Activos:                  {len(df_activos)}")
print(f"Scaffolds genéricos únicos: {len(scaffolds_activos)}")
print()
print("Ejemplos de scaffolds genéricos:")
for scf in list(scaffolds_activos)[:3]:
    print(f"  {scf}")


Activos:                  29
Scaffolds genéricos únicos: 7

Ejemplos de scaffolds genéricos:
  CC1CCC(C2CCCC2)C(C)C1
  C1CCCCC1
  CC1CCC2CCCCC2C1


---
## 3. Selección de decoys desde Dark Chemical Matter (DCM)

Los **Dark Chemical Matter** son compuestos que han sido testados en múltiples
ensayos de HTS pero nunca han mostrado actividad — son negativos confiables.
Son ideales como decoys porque:

1. **Tienen actividad confirmada como INACTIVOS** en múltiples targets
2. Cubren un espacio químico diverso
3. Ya vienen preparados con scaffolds genéricos calculados (columna `scf`)

Fuente: [SmartDock — Koch Group](https://github.com/kochgroup/smartdock)


In [5]:
# ── Descargar y cargar Dark Chemical Matter (DCM) ────────────────────────────
URL_DCM = ("https://raw.githubusercontent.com/kochgroup/smartdock/"
           "main/.src/DCM_prepared.csv")

ARCHIVO_DCM = 'DCM_prepared.csv'
if os.path.exists(ARCHIVO_DCM):
    df_dcm = pd.read_csv(ARCHIVO_DCM)
    print(f"✅ DCM cargado localmente: {len(df_dcm)} moléculas")
else:
    print("Descargando Dark Chemical Matter desde GitHub (Koch Group)...")
    r = requests.get(URL_DCM, timeout=120)
    if r.status_code == 200:
        with open(ARCHIVO_DCM, 'wb') as fh:
            fh.write(r.content)
        df_dcm = pd.read_csv(ARCHIVO_DCM)
        print(f"✅ DCM descargado y cargado: {len(df_dcm)} moléculas")
    else:
        raise ConnectionError(f"No se pudo descargar DCM: HTTP {r.status_code}")

# El CSV ya tiene la columna 'scf' (scaffold genérico) calculada
print(f"   Columnas: {df_dcm.columns.tolist()}")
print()
print(df_dcm.head(3)[['molecule_chembl_id','smiles','molwt','scf']].to_string(index=False))


Descargando Dark Chemical Matter desde GitHub (Koch Group)...
✅ DCM descargado y cargado: 119185 moléculas
   Columnas: ['InChI_Key', 'set', 'smiles', 'molecule_chembl_id', 'molwt', 'scf', 'id']

molecule_chembl_id                                              smiles   molwt                                    scf
     CHEMBL1605650 C[C@@H]1CCC[C@H](C)N1NC(=O)c1ccc(Cl)c(S(N)(=O)=O)c1 345.852 CC(CC1C(C)CCCC1C)C1CCC(C)C(C(C)(C)C)C1
     CHEMBL1713382                         Cc1ccc(N=C(N)N=C(N)N)c(C)c1 205.265                CC(C)CC(C)CC1CCC(C)CC1C
     CHEMBL1437772                          CC(=O)N1c2ccccc2Sc2ccccc21 241.315              CC(C)C1C2CCCCC2CC2CCCCC21


In [6]:
# ── Calcular propiedades de activos para el filtro de matching ────────────────
from rdkit.Chem import Descriptors

def calc_props(mol):
    return {
        'MW':   Descriptors.MolWt(mol),
        'LogP': Descriptors.MolLogP(mol),
        'TPSA': Descriptors.TPSA(mol),
        'HBD':  Descriptors.NumHDonors(mol),
        'HBA':  Descriptors.NumHAcceptors(mol),
    }

# Propiedades de los activos ChEMBL
props_activos = []
for smi in df_activos['std_smiles']:
    mol = Chem.MolFromSmiles(smi)
    if mol:
        props_activos.append(calc_props(mol))
df_props_act = pd.DataFrame(props_activos)

# Rangos ±20% de tolerancia sobre los activos
rangos = {}
for col in ['MW','LogP','TPSA','HBD','HBA']:
    mn, mx   = df_props_act[col].min(), df_props_act[col].max()
    margen   = (mx - mn) * 0.20
    rangos[col] = (mn - margen, mx + margen)

print("Rangos de propiedades de activos (±20%):")
for k, (lo, hi) in rangos.items():
    print(f"  {k:<6}: [{lo:.2f}, {hi:.2f}]")

# Calcular propiedades del DCM
# DCM ya tiene molwt → usarla directamente, calcular el resto
print("\nCalculando propiedades del DCM...")
dcm_props = []
for smi in tqdm(df_dcm['smiles'], desc='Props DCM'):
    mol = Chem.MolFromSmiles(str(smi))
    if mol:
        dcm_props.append(calc_props(mol))
    else:
        dcm_props.append({k: np.nan for k in ['MW','LogP','TPSA','HBD','HBA']})

df_dcm_props = pd.DataFrame(dcm_props)
df_dcm = pd.concat([df_dcm.reset_index(drop=True), df_dcm_props], axis=1)

# Filtrar por propiedades fisicoquímicas similares a los activos
mask = pd.Series([True] * len(df_dcm))
for col, (lo, hi) in rangos.items():
    mask = mask & (df_dcm[col] >= lo) & (df_dcm[col] <= hi)

df_dcm_filt = df_dcm[mask].dropna(subset=['smiles']).reset_index(drop=True)
print(f"\nDCM tras filtro de propiedades: {len(df_dcm_filt)} / {len(df_dcm)} moléculas")


Rangos de propiedades de activos (±20%):
  MW    : [158.29, 900.85]
  LogP  : [-8.76, 7.51]
  TPSA  : [-41.94, 426.09]
  HBD   : [-1.00, 13.00]
  HBA   : [-0.40, 16.40]

Calculando propiedades del DCM...


Props DCM:   0%|          | 0/119185 [00:00<?, ?it/s]


DCM tras filtro de propiedades: 119060 / 119185 moléculas


In [7]:
# ── Filtrar por scaffold genérico distinto a los activos ─────────────────────
# DCM ya tiene la columna 'scf' con el scaffold genérico calculado
# No necesitamos recalcularla — ahorramos tiempo

# Excluir compuestos cuyo scaffold coincida con algún activo
mask_nuevo_scf = ~df_dcm_filt['scf'].isin(scaffolds_activos)
df_decoys_cand = df_dcm_filt[mask_nuevo_scf].reset_index(drop=True)

# Un representante por scaffold (máxima diversidad estructural)
df_decoys_cand = (df_decoys_cand
                  .dropna(subset=['scf'])
                  .drop_duplicates(subset='scf')
                  .reset_index(drop=True))

# Limitar a máximo 2× el número de activos
N_DECOYS_MAX = min(len(df_activos) * 2, len(df_decoys_cand))
df_decoys    = df_decoys_cand.sample(N_DECOYS_MAX, random_state=42).reset_index(drop=True)

# Normalizar nombres de columnas para el resto del notebook
df_decoys = df_decoys.rename(columns={
    'smiles':           'std_smiles',
    'molecule_chembl_id':'molecule_id',
    'scf':              'scaffold_generico',
})

print(f"Activos:              {len(df_activos)}")
print(f"DCM disponibles:      {len(df_dcm_filt)}")
print(f"Decoys seleccionados: {len(df_decoys)}  (un scaffold único por compuesto)")
print(f"Ratio decoys/activos: {len(df_decoys)/len(df_activos):.1f}x")
print()
print(f"Ejemplos de scaffolds DCM seleccionados:")
for scf in df_decoys['scaffold_generico'].head(3).tolist():
    print(f"  {scf}")


Activos:              29
DCM disponibles:      119060
Decoys seleccionados: 58  (un scaffold único por compuesto)
Ratio decoys/activos: 2.0x

Ejemplos de scaffolds DCM seleccionados:
  CC(CC1CCC(C(C)(C)CCC2CCCCC2)CC1)CC1CCCCC1C1CCCCC1
  CC1CC(C(C)C2CCC2(C)C(C)CCC2CCCC3CCCCC32)CC1C
  CC1CCC(CC2C3CC(C)CCC3CC2C2CCCCC2)CC1


In [8]:
# ── Seleccionar 4 decoys por cada activo (matching 1:4) ─────────────────────
import numpy as np

def seleccionar_decoys_por_activo(df_activos, df_decoys_candidatos,
                                   n_decoys_por_activo=4, random_state=42):
    """
    Selecciona n_decoys_por_activo decoys para cada activo usando
    matching por propiedades fisicoquímicas más cercanas.

    Para cada activo:
      1. Calcula la distancia euclídea normalizada en el espacio MW/LogP/TPSA
      2. Toma los n_decoys_por_activo decoys más cercanos que aún no hayan
         sido asignados (sin reemplazamiento entre activos)

    Parámetros
    ----------
    df_activos            : DataFrame con columna 'std_smiles'
    df_decoys_candidatos  : DataFrame con columnas 'std_smiles' y 'scaffold_generico'
    n_decoys_por_activo   : int — número de decoys por activo (default 4)
    random_state          : int — semilla para reproducibilidad

    Retorna
    -------
    df_decoys_sel : DataFrame con los decoys seleccionados y columna 'activo_match'
    """
    from sklearn.preprocessing import StandardScaler
    from rdkit.Chem import Descriptors

    np.random.seed(random_state)

    def props(smiles):
        mol = Chem.MolFromSmiles(str(smiles))
        if mol is None:
            return None
        return np.array([
            Descriptors.MolWt(mol),
            Descriptors.MolLogP(mol),
            Descriptors.TPSA(mol),
            Descriptors.NumHDonors(mol),
            Descriptors.NumHAcceptors(mol),
        ], dtype=float)

    # Calcular propiedades de activos
    props_act = []
    ids_act   = []
    for _, row in df_activos.iterrows():
        p = props(row['std_smiles'])
        if p is not None:
            props_act.append(p)
            ids_act.append(row.get('molecule_chembl_id', f'activo_{_}'))

    # Calcular propiedades de decoys candidatos
    props_dec = []
    idx_dec   = []
    for i, row in df_decoys_candidatos.iterrows():
        p = props(row['std_smiles'])
        if p is not None:
            props_dec.append(p)
            idx_dec.append(i)

    X_act = np.array(props_act)
    X_dec = np.array(props_dec)

    # Escalar conjuntamente
    scaler  = StandardScaler()
    X_todos = scaler.fit_transform(np.vstack([X_act, X_dec]))
    X_act_s = X_todos[:len(X_act)]
    X_dec_s = X_todos[len(X_act):]

    disponibles  = set(range(len(X_dec_s)))
    seleccionados = []

    for i, (act_vec, act_id) in enumerate(zip(X_act_s, ids_act)):
        if len(disponibles) < n_decoys_por_activo:
            print(f"⚠️  Pocos decoys disponibles ({len(disponibles)}) para activo {act_id}")
            candidatos_idx = list(disponibles)
        else:
            # Distancia euclídea a todos los decoys disponibles
            dists   = np.linalg.norm(X_dec_s[list(disponibles)] - act_vec, axis=1)
            disp_arr = np.array(list(disponibles))
            # Top n_decoys_por_activo más cercanos
            top_idx = disp_arr[np.argsort(dists)[:n_decoys_por_activo]]
            candidatos_idx = top_idx.tolist()

        # Marcar como usados
        for ci in candidatos_idx:
            disponibles.discard(ci)

        # Guardar con referencia al activo
        for ci in candidatos_idx:
            fila = df_decoys_candidatos.iloc[idx_dec[ci]].copy()
            fila['activo_match'] = act_id
            seleccionados.append(fila)

    df_sel = pd.DataFrame(seleccionados).reset_index(drop=True)
    return df_sel

In [9]:
# ── Verificar columnas disponibles ───────────────────────────────────────────
print("Columnas en df_activos:")
print(f"  {df_activos.columns.tolist()}")
print()
print("Columnas en df_decoys_cand:")
print(f"  {df_decoys_cand.columns.tolist()}")

Columnas en df_activos:
  ['canonical_smiles', 'molecule_chembl_id', 'pchembl_value', 'standard_type', 'standard_units', 'standard_value', 'type', 'units', 'value', 'pActividad', 'mol_weight', 'std_smiles', 'cur_error', 'pdbqt', 'scaffold_murcko', 'scaffold_generico']

Columnas en df_decoys_cand:
  ['InChI_Key', 'set', 'smiles', 'molecule_chembl_id', 'molwt', 'scf', 'id', 'MW', 'LogP', 'TPSA', 'HBD', 'HBA']


In [10]:
# ── Normalizar columnas antes de llamar la función ───────────────────────────
# Activos: asegurar que tiene 'std_smiles'
if 'std_smiles' not in df_activos.columns:
    col_smi_act = next((c for c in df_activos.columns
                        if 'smiles' in c.lower()), None)
    if col_smi_act:
        df_activos = df_activos.rename(columns={col_smi_act: 'std_smiles'})
        print(f"✅ Activos: renombrado '{col_smi_act}' → 'std_smiles'")

# Decoys: asegurar que tiene 'std_smiles'
if 'std_smiles' not in df_decoys_cand.columns:
    col_smi_dec = next((c for c in df_decoys_cand.columns
                        if 'smiles' in c.lower()), None)
    if col_smi_dec:
        df_decoys_cand = df_decoys_cand.rename(columns={col_smi_dec: 'std_smiles'})
        print(f"✅ Decoys: renombrado '{col_smi_dec}' → 'std_smiles'")

# Decoys: asegurar que tiene 'molecule_chembl_id' o 'molecule_id'
if 'molecule_chembl_id' not in df_decoys_cand.columns and \
   'molecule_id' not in df_decoys_cand.columns:
    col_id = next((c for c in df_decoys_cand.columns
                   if 'id' in c.lower()), None)
    if col_id:
        df_decoys_cand = df_decoys_cand.rename(columns={col_id: 'molecule_id'})
        print(f"✅ Decoys: renombrado '{col_id}' → 'molecule_id'")

print()
print(f"Columnas activos:  {df_activos.columns.tolist()}")
print(f"Columnas decoys:   {df_decoys_cand.columns.tolist()}")
print()

# ── Ejecutar selección ────────────────────────────────────────────────────────
df_decoys = seleccionar_decoys_por_activo(
    df_activos,
    df_decoys_cand,
    n_decoys_por_activo=4,
    random_state=42
)

✅ Decoys: renombrado 'smiles' → 'std_smiles'

Columnas activos:  ['canonical_smiles', 'molecule_chembl_id', 'pchembl_value', 'standard_type', 'standard_units', 'standard_value', 'type', 'units', 'value', 'pActividad', 'mol_weight', 'std_smiles', 'cur_error', 'pdbqt', 'scaffold_murcko', 'scaffold_generico']
Columnas decoys:   ['InChI_Key', 'set', 'std_smiles', 'molecule_chembl_id', 'molwt', 'scf', 'id', 'MW', 'LogP', 'TPSA', 'HBD', 'HBA']



---
## 4. Preparación de decoys: conformeros 3D y PDBQT


In [14]:
# ── Función de generación de conformero + conversión a PDBQT ─────────────────
def preparar_ligando(smiles, mol_id):
    """
    Genera el conformero 3D y convierte a formato PDBQT con Meeko.
    Retorna el string PDBQT o None si falla.
    """
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None: return None
        mol = Chem.AddHs(mol)
        result = AllChem.EmbedMolecule(mol, AllChem.ETKDGv3())
        if result != 0:
            result = AllChem.EmbedMolecule(mol,
                AllChem.ETKDGv3(), randomSeed=42)
        if result != 0: return None
        AllChem.MMFFOptimizeMolecule(mol)
        preparator = MoleculePreparation()
        preparator.prepare(mol)
        return preparator.write_pdbqt_string()
    except Exception as e:
        return None

# Preparar decoys
print(f"Preparando {len(df_decoys)} decoys (conformeros 3D + PDBQT)...")
pdbqts_decoys = []
for _, row in tqdm(df_decoys.iterrows(), total=len(df_decoys)):
    pdbqt = preparar_ligando(row['std_smiles'], row['id'])
    pdbqts_decoys.append(pdbqt)

df_decoys['pdbqt'] = pdbqts_decoys
df_decoys_validos = df_decoys[df_decoys['pdbqt'].notna()].reset_index(drop=True)

print(f"\n✅ Decoys preparados: {len(df_decoys_validos)} / {len(df_decoys)}")
print(f"   Fallidos: {len(df_decoys) - len(df_decoys_validos)}")


Preparando 116 decoys (conformeros 3D + PDBQT)...


  0%|          | 0/116 [00:00<?, ?it/s]

/home/dfvic/.local/lib/python3.14/site-packages/meeko/preparation.py:693: DeprecationWarning: MoleculePreparation.write_pdbqt_string() is deprecated in Meeko v0.5. Pass the MoleculeSetup instance to PDBQTWriterLegacy.write_string(). MoleculePreparation.prepare() returns a list of MoleculeSetup instances.
  warnings.warn(msg, DeprecationWarning)
/home/dfvic/.local/lib/python3.14/site-packages/meeko/preparation.py:467: DeprecationWarning: MoleculePreparation.setup is deprecated in Meeko v0.5. MoleculePreparation.prepare() returns a list of MoleculeSetup instances.
  warnings.warn(msg, DeprecationWarning)



✅ Decoys preparados: 116 / 116
   Fallidos: 0


---
## 5. Docking en batch de decoys

Usamos exactamente los mismos parámetros que en NB-ML-02 (misma caja, misma
exhaustiveness) para que los resultados sean comparables.


In [ ]:
# ── Función de docking individual ────────────────────────────────────────────
def vina_docking_decoy(mol_id, pdbqt_str, output_dir,
                       center=None, box_size=None,
                       exhaustiveness=8, n_poses=5):
    """Corre Vina para un decoy y guarda el SDF resultado.

    Si center/box_size son None, los carga automáticamente desde
    estructuras/docking_params.json.
    """
    out_sdf = os.path.join(output_dir, f'{mol_id}.sdf')
    if os.path.exists(out_sdf):
        return True   # ya calculado

    # ── Cargar parámetros de docking desde JSON si no se pasaron ─────────────
    if center is None or box_size is None:
        params_path = os.path.join(protein_directory, 'docking_params.json')
        with open(params_path) as fh:
            dp = json.load(fh)
        center   = np.array(dp['pocket_center'])
        box_size = np.array(dp['ligand_box'])

    pdbqt_path = os.path.join(output_dir, f'{mol_id}.pdbqt')
    try:
        with open(pdbqt_path, 'w') as f:
            f.write(pdbqt_str)

        v = Vina(sf_name='vina', verbosity=0)
        v.set_receptor(os.path.join(pdbqt_directory, f'{PDB_ID}.pdbqt'))
        v.set_ligand_from_file(pdbqt_path)
        v.compute_vina_maps(center=center.tolist(),
                            box_size=box_size.tolist())
        v.dock(exhaustiveness=exhaustiveness, n_poses=n_poses)
        v.write_poses(out_sdf, n_poses=n_poses, overwrite=True)
        return True
    except Exception as e:
        return False
    finally:
        if os.path.exists(pdbqt_path):
            os.remove(pdbqt_path)


In [ ]:
# ── Docking en batch ─────────────────────────────────────────────────────────
from joblib import Parallel, delayed

ya_procesados = set(
    os.path.splitext(os.path.basename(f))[0]
    for f in __import__('glob').glob(f'{decoys_directory}/*.sdf')
)
pendientes = df_decoys_validos[
    ~df_decoys_validos['id'].isin(ya_procesados)
]

print(f"Decoys totales:     {len(df_decoys_validos)}")
print(f"Ya procesados:      {len(ya_procesados)}")
print(f"Pendientes:         {len(pendientes)}")

if len(pendientes) == 0:
    print("✅ Todos los dockings ya están calculados")
else:
    n_jobs = min(2, os.cpu_count() or 1)
    print(f"\nIniciando docking ({n_jobs} workers, exhaustiveness=8)...")

    resultados = Parallel(n_jobs=n_jobs, prefer='threads')(
        delayed(vina_docking_decoy)(
            row.id, row.pdbqt, decoys_directory
        )
        for _, row in tqdm(pendientes.iterrows(), total=len(pendientes),
                           desc='Docking decoys')
    )
    exitosos = sum(resultados)
    print(f"\n✅ Docking completado: {exitosos} / {len(pendientes)}")


In [20]:
pendientes

,InChI_Key,set,std_smiles,molecule_chembl_id,molwt,scf,id,MW,LogP,TPSA,HBD,HBA,activo_match,pdbqt
0,PZUGUHSUBUNLKU-UHFFFAOYSA-N,PubChem,O=C(CN1CCCC1)NC(c1ccccc1)c1ccccc1,CHEMBL1617022,294.398,CC(CC1CCCC1)CC(C1CCCCC1)C1CCCCC1,DCM-48045,294.398,2.98800,32.34,1,2,CHEMBL36409,REMARK SMILES O=C(CN1CCCC1)NC(c1ccccc1)c1ccccc...
1,LWLUTXFFXVFYMD-UHFFFAOYSA-N,PubChem,Cc1ccccc1CN(C)C(C)C(=O)NCc1ccccc1,CHEMBL1418498,296.414,CC1CCCCC1CC(C)C(C)C(C)CCC1CCCCC1,DCM-61372,296.414,3.13172,32.34,1,2,CHEMBL36409,REMARK SMILES Cc1ccccc1CN(C)C(C)C(=O)NCc1ccccc...
2,BQHVIMLXURNCBU-UHFFFAOYSA-N,Novartis,CCN(CC)CCNC(=O)c1ccc(Cl)c(Cl)c1,not found,289.206,CCC(CC)CCCC(C)C1CCC(C)C(C)C1,DCM-113596,289.206,3.06500,32.34,1,2,CHEMBL36409,REMARK SMILES CCN(CC)CCNC(=O)c1ccc(Cl)c(Cl)c1\...
3,DFWFHKKMVICNGN-UHFFFAOYSA-N,PubChem,CCCN(CC1CC1)C(=S)NC(=O)c1ccccc1C,CHEMBL2144342,290.432,CCCC(CC1CC1)C(C)CC(C)C1CCCCC1C,DCM-98408,290.432,3.13172,32.34,1,2,CHEMBL36409,REMARK SMILES CCCN(CC1CC1)C(=S)NC(=O)c1ccccc1C...
4,ZOAQYHQNVPIKDS-UHFFFAOYSA-N,PubChem,O=C(Cc1ccc(Cl)c(Cl)c1)Nc1ccccc1N1CCCC1,CHEMBL1732755,349.261,CC(CC1CCC(C)C(C)C1)CC1CCCCC1C1CCCC1,DCM-101252,349.261,4.77480,32.34,1,2,CHEMBL285056,REMARK SMILES O=C(Cc1ccc(Cl)c(Cl)c1)Nc1ccccc1N...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
111,LEGBHXGPDYDBEO-UHFFFAOYSA-N,PubChem,Cc1ccc(NC2=CC(=O)CC(C)(C)C2)cc1Cl,CHEMBL1448643,263.768,CC1CC(CC2CCC(C)C(C)C2)CC(C)(C)C1,DCM-50323,263.768,4.33332,29.10,1,2,CHEMBL422402,REMARK SMILES Cc1ccc(NC2=CC(=O)CC(C)(C)C2)cc1C...
112,XRCYKIVYVCRJAM-BMSKXHDZSA-N,PubChem,O=C(CCCCO)CCCCCCCC[C@H]1N[C@H](CO)[C@H](O)[C@@...,CHEMBL3198442,507.621,CCCCCC(C)CCCCCCCCC1CC(CC)C(C)C1CC1CC(CC)C(C)C(...,DCM-15380,507.621,-1.28240,189.17,8,11,CHEMBL1791436,REMARK SMILES O=C(CCCCO)CCCCCCCC[C@H]1N[C@H](C...
113,CCFUYOZYVIRWFQ-AXKRAVOISA-N,PubChem,Nc1nc(=O)c2ncn([C@@H]3O[C@H](CNC(=O)[C@H](N)Cc...,CHEMBL1897670,539.553,CC1CC(C)C2CCC(C3CC(CCC(C)C(C)CC4CCCC4CCCC4CCCC...,DCM-112073,539.553,-1.62200,221.45,6,11,CHEMBL1791436,REMARK SMILES Nc1nc(=O)c2ncn([C@@H]3O[C@H](CNC...
114,ZPHBZEQOLSRPAK-XLCYBJAPSA-M,PubChem,CC(C)C[C@H](NP(=O)([O-])O[C@@H]1O[C@@H](C)[C@H...,not found,542.502,CC(C)CC(CC(C)(C)CC1CC(C)C(C)C(C)C1C)C(C)CC(CC1...,DCM-17154,542.502,-0.40340,213.50,7,9,CHEMBL1791436,REMARK SMILES CC(C)C[C@H](NP(=O)([O-])O[C@@H]1...


---
## 6. Fingerprints ProLIF para decoys

Calculamos el mismo tipo de fingerprint de interacciones que se usó para los activos.


In [ ]:
# ── Cargar proteína para ProLIF ──────────────────────────────────────────────
rdkit_prot = Chem.MolFromPDBFile(
    f'{protein_directory}/{PDB_ID}_prep.pdb',
    removeHs=False, sanitize=False
)
if rdkit_prot is None:
    rdkit_prot = Chem.MolFromPDBFile(
        f'{protein_directory}/{PDB_ID}_a.pdb',
        removeHs=False, sanitize=False
    )
protein_mol = plf.Molecule(rdkit_prot)
print(f"✅ Proteína cargada: {PDB_ID}")


In [ ]:
# ── Función ProLIF (igual que NB-ML-02) ──────────────────────────────────────
def get_prolif_fp(sdf_path, mol_id, protein):
    """
    Calcula el fingerprint de interacciones ProLIF para la mejor pose (pose 1).
    Retorna un DataFrame de 1 fila o None si falla.
    """
    try:
        poses = plf.sdf_supplier(sdf_path)
        if not poses: return None
        fp = plf.Fingerprint(vicinity_cutoff=8.0, count=True)
        fp.run_from_iterable([poses[0]], protein)
        df_fp = fp.to_dataframe()
        df_fp.index = [mol_id]
        return df_fp
    except Exception:
        return None


In [ ]:
# ── Calcular ProLIF para todos los decoys ────────────────────────────────────
import glob

sdfs_decoys = glob.glob(f'{decoys_directory}/*.sdf')
print(f"SDFs de decoys encontrados: {len(sdfs_decoys)}")

fps_decoys = []
for sdf_path in tqdm(sdfs_decoys, desc='ProLIF decoys'):
    mol_id = os.path.splitext(os.path.basename(sdf_path))[0]
    fp = get_prolif_fp(sdf_path, mol_id, protein_mol)
    if fp is not None:
        fps_decoys.append(fp)

if fps_decoys:
    fps_decoys_df = pd.concat(fps_decoys)
    fps_decoys_df = fps_decoys_df.fillna(0)
    print(f"\n✅ ProLIF decoys: {fps_decoys_df.shape[0]} moléculas × {fps_decoys_df.shape[1]} interacciones")
else:
    print("⚠️  No se pudieron calcular fingerprints para los decoys")
    fps_decoys_df = pd.DataFrame()


---
## 7. DataFrame combinado: activos + decoys con etiqueta de actividad


In [ ]:
# ── Cargar fingerprints de activos ────────────────────────────────────────────
URL_FPS = ("https://raw.githubusercontent.com/FelPVic/curso_datascience/"
           "main/files/fps_activos.csv")
ARCHIVO_FPS = 'fps_activos.csv'

if os.path.exists(ARCHIVO_FPS):
    fps_raw = pd.read_csv(ARCHIVO_FPS)
else:
    fps_raw = pd.read_csv(URL_FPS)

# El CSV tiene 3 filas de cabecera: protein, interaction, mol
# Las filas de datos empiezan en la fila 2 (índice 2)
fps_header_prot = fps_raw.iloc[0]          # residuo proteína
fps_header_inter = fps_raw.iloc[1]         # tipo interacción
fps_activos_df   = fps_raw.iloc[2:].copy() # datos reales

# Renombrar columna de ID
fps_activos_df = fps_activos_df.rename(columns={fps_activos_df.columns[0]: 'mol_id'})
fps_activos_df = fps_activos_df.set_index('mol_id')
fps_activos_df = fps_activos_df.apply(pd.to_numeric, errors='coerce').fillna(0)

fps_activos_df['activity'] = 1   # activos = 1
print(f"✅ Fingerprints activos: {fps_activos_df.shape}")


In [ ]:
# ── Preparar fingerprints de decoys ──────────────────────────────────────────
if not fps_decoys_df.empty:
    fps_decoys_df_labeled = fps_decoys_df.copy()
    fps_decoys_df_labeled['activity'] = 0   # decoys = 0

    # Alinear columnas: usar solo las columnas que están en ambos
    cols_comunes = [c for c in fps_activos_df.columns
                    if c in fps_decoys_df_labeled.columns
                    and c != 'activity']

    df_activos_alin = fps_activos_df[cols_comunes + ['activity']].copy()
    df_decoys_alin  = fps_decoys_df_labeled[cols_comunes + ['activity']].copy()

    # Combinar
    df_combined = pd.concat([df_activos_alin, df_decoys_alin])
    df_combined = df_combined.fillna(0).reset_index()
    df_combined = df_combined.rename(columns={'index': 'mol_id'})

    print(f"Dataset combinado: {df_combined.shape}")
    print(f"  Activos:  {(df_combined['activity']==1).sum()}")
    print(f"  Decoys:   {(df_combined['activity']==0).sum()}")
    print(f"  Features: {len(cols_comunes)}")
else:
    print("⚠️  No hay fingerprints de decoys — revisar el docking")
    df_combined = fps_activos_df.reset_index()


---
## 8. División del dataset: 3 estrategias de split

Evaluamos tres estrategias distintas para medir la robustez del modelo:
- **Random estratificado**: división aleatoria manteniendo proporción activos/decoys
- **Scaffold split**: test contiene scaffolds no vistos en training
- **Fingerprint split**: test contiene moléculas disimilares al training


In [ ]:
# ── Split 1: Random estratificado ───────────────────────────────────────────
from sklearn.model_selection import train_test_split

# Separar features y etiqueta
feature_cols = [c for c in df_combined.columns
                if c not in ['mol_id', 'activity', 'smiles']]
X = df_combined[feature_cols].values
y = df_combined['activity'].values

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("SPLIT 1 — Random estratificado")
print(f"  Train: {len(y_train_r)}  (activos: {y_train_r.sum()}, decoys: {(y_train_r==0).sum()})")
print(f"  Test:  {len(y_test_r)}   (activos: {y_test_r.sum()}, decoys: {(y_test_r==0).sum()})")
print(f"  Prop. activos train: {y_train_r.mean():.3f}")
print(f"  Prop. activos test:  {y_test_r.mean():.3f}")


In [ ]:
# ── Split 2: Scaffold split ───────────────────────────────────────────────────
# Asignar scaffold a cada molécula del dataset combinado
id_to_scaffold = {}
for _, row in df_activos.iterrows():
    mid = row['molecule_chembl_id']
    id_to_scaffold[mid] = row.get('scaffold_generico', '')
for _, row in df_decoys.iterrows():
    mid = row['molecule_id']
    id_to_scaffold[mid] = row.get('scaffold_generico', '')

df_combined['scaffold'] = df_combined['mol_id'].map(id_to_scaffold).fillna('unknown')

# Agrupar por scaffold y hacer el split
scaffolds_unicos = df_combined['scaffold'].unique()
scf_train, scf_test = train_test_split(scaffolds_unicos,
                                        test_size=0.2, random_state=42)
mask_train_s = df_combined['scaffold'].isin(scf_train)
mask_test_s  = df_combined['scaffold'].isin(scf_test)

X_train_s  = df_combined.loc[mask_train_s, feature_cols].values
X_test_s   = df_combined.loc[mask_test_s,  feature_cols].values
y_train_s  = df_combined.loc[mask_train_s, 'activity'].values
y_test_s   = df_combined.loc[mask_test_s,  'activity'].values

print("SPLIT 2 — Scaffold split")
print(f"  Train: {len(y_train_s)}  (activos: {y_train_s.sum()})")
print(f"  Test:  {len(y_test_s)}   (activos: {y_test_s.sum()})")
print(f"  Scaffolds train: {len(scf_train)}  |  test: {len(scf_test)}")


In [ ]:
# ── Split 3: Fingerprint split (basado en similitud Tanimoto) ─────────────────
from sklearn.cluster import KMeans

# Calcular Morgan fingerprints para clustering
def morgan_fp(smiles, n_bits=2048):
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None: return None
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=n_bits)
    arr = np.zeros(n_bits, dtype=np.uint8)
    Chem.DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

# SMILES por mol_id
id_to_smiles = {}
for _, r in df_activos.iterrows():
    id_to_smiles[r['molecule_chembl_id']] = r['std_smiles']
for _, r in df_decoys.iterrows():
    id_to_smiles[r['molecule_id']] = r['std_smiles']

fps_cluster = []
valid_idx = []
for i, mid in enumerate(df_combined['mol_id']):
    smi = id_to_smiles.get(mid, '')
    fp = morgan_fp(smi)
    if fp is not None:
        fps_cluster.append(fp)
        valid_idx.append(i)

X_fps_cluster = np.array(fps_cluster)

# Clustering: 80% clusters → train, 20% → test
n_clusters = max(10, len(X_fps_cluster) // 5)
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=5)
cluster_labels = kmeans.fit_predict(X_fps_cluster)

clusters_unicos = np.unique(cluster_labels)
cl_train, cl_test = train_test_split(clusters_unicos, test_size=0.2, random_state=42)

mask_train_fp = np.array([cluster_labels[valid_idx.index(i)] in cl_train
                            if i in valid_idx else True
                            for i in range(len(df_combined))])
mask_test_fp  = ~mask_train_fp

X_train_fp = df_combined.loc[mask_train_fp, feature_cols].values
X_test_fp  = df_combined.loc[mask_test_fp,  feature_cols].values
y_train_fp = df_combined.loc[mask_train_fp, 'activity'].values
y_test_fp  = df_combined.loc[mask_test_fp,  'activity'].values

print("SPLIT 3 — Fingerprint split (clustering Morgan)")
print(f"  Train: {len(y_train_fp)}  (activos: {y_train_fp.sum()})")
print(f"  Test:  {len(y_test_fp)}   (activos: {y_test_fp.sum()})")


---
## 9. Entrenamiento XGBoost sobre los 3 splits

Métricas de interés en QSAR con desbalance:
- **F1**: balance entre precisión y recall
- **MCC**: el más robusto al desbalance (Matthews Correlation Coefficient)
- **Balanced Accuracy**: accuracy corregida por el desbalance de clases


In [ ]:
# ── Función de evaluación ────────────────────────────────────────────────────
from sklearn.metrics import (f1_score, matthews_corrcoef,
                              balanced_accuracy_score,
                              classification_report,
                              ConfusionMatrixDisplay, confusion_matrix)

def entrenar_evaluar_xgb(X_tr, y_tr, X_te, y_te, nombre_split):
    """Entrena XGBoost y retorna métricas + modelo."""
    scale_pos = float((y_tr==0).sum() / y_tr.sum()) if y_tr.sum() > 0 else 1.0

    modelo = xgb.XGBClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos,
        use_label_encoder=False,
        eval_metric='logloss',
        random_state=42,
        n_jobs=-1
    )
    modelo.fit(X_tr, y_tr,
               eval_set=[(X_te, y_te)],
               verbose=False)

    y_pred  = modelo.predict(X_te)
    y_proba = modelo.predict_proba(X_te)[:,1]

    f1  = f1_score(y_te, y_pred, zero_division=0)
    mcc = matthews_corrcoef(y_te, y_pred)
    ba  = balanced_accuracy_score(y_te, y_pred)

    print(f"\n{'─'*55}")
    print(f"  SPLIT: {nombre_split}")
    print(f"{'─'*55}")
    print(f"  F1 Score:          {f1:.4f}")
    print(f"  MCC:               {mcc:.4f}")
    print(f"  Balanced Accuracy: {ba:.4f}")
    print()
    print(classification_report(y_te, y_pred,
                                 target_names=['Decoy','Activo'],
                                 zero_division=0))

    return modelo, {'Split': nombre_split, 'F1': f1, 'MCC': mcc,
                    'BalancedAcc': ba}, y_pred, y_proba

resultados_splits = []
modelos_splits    = {}
predicciones      = {}


In [ ]:
# ── Split 1: Random ──────────────────────────────────────────────────────────
m1, res1, yp1, ypr1 = entrenar_evaluar_xgb(
    X_train_r, y_train_r, X_test_r, y_test_r, 'Random estratificado')
resultados_splits.append(res1)
modelos_splits['Random'] = m1
predicciones['Random']   = (y_test_r, yp1, ypr1)


In [ ]:
# ── Split 2: Scaffold ────────────────────────────────────────────────────────
m2, res2, yp2, ypr2 = entrenar_evaluar_xgb(
    X_train_s, y_train_s, X_test_s, y_test_s, 'Scaffold split')
resultados_splits.append(res2)
modelos_splits['Scaffold'] = m2
predicciones['Scaffold']   = (y_test_s, yp2, ypr2)


In [ ]:
# ── Split 3: Fingerprint ──────────────────────────────────────────────────────
m3, res3, yp3, ypr3 = entrenar_evaluar_xgb(
    X_train_fp, y_train_fp, X_test_fp, y_test_fp, 'Fingerprint split')
resultados_splits.append(res3)
modelos_splits['Fingerprint'] = m3
predicciones['Fingerprint']   = (y_test_fp, yp3, ypr3)


In [ ]:
# ── Tabla comparativa de los 3 splits ────────────────────────────────────────
df_resultados_splits = pd.DataFrame(resultados_splits).set_index('Split')
print("COMPARACIÓN DE LOS 3 SPLITS — XGBoost")
print("=" * 50)
print(df_resultados_splits.round(4).to_string())

# Gráfico comparativo
fig, axes = plt.subplots(1, 3, figsize=(13, 5))
metricas_plot = ['F1', 'MCC', 'BalancedAcc']
colores_split = ['#3d6b99','#27ae60','#f97316']

for ax, metrica in zip(axes, metricas_plot):
    vals = df_resultados_splits[metrica]
    bars = ax.bar(vals.index.tolist(), vals,
                  color=colores_split, alpha=0.85, edgecolor='white')
    ax.set_ylabel(metrica, fontsize=11)
    ax.set_title(metrica, fontsize=12, fontweight='bold')
    ax.set_ylim(0, 1.1)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.02,
                f'{val:.3f}', ha='center', fontsize=10)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('XGBoost — Activos vs Decoys (3 splits)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('results/comparacion_splits.png', dpi=130, bbox_inches='tight')
plt.show()


In [ ]:
# ── Matrices de confusión ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (nombre, (y_true, y_pred, _)) in zip(axes, predicciones.items()):
    cm = confusion_matrix(y_true, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=['Decoy','Activo']).plot(
        ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(f'{nombre}', fontsize=11)
plt.suptitle('Matrices de confusión — XGBoost', fontsize=12)
plt.tight_layout()
plt.savefig('results/matrices_confusion.png', dpi=130, bbox_inches='tight')
plt.show()


---
## 10. Predicción sobre muestra de DCM no usada en entrenamiento

Seleccionamos 30 moléculas de DCM que **no** se usaron como decoys de entrenamiento,
las acoplamos y clasificamos con los 3 modelos para ver cuántas el modelo identifica
como potencialmente activas (falsos negativos del DCM → candidatos interesantes).


In [ ]:
# ── Seleccionar 30 moléculas DCM no usadas en entrenamiento ──────────────────
ids_usados_como_decoy = set(df_decoys['molecule_id'].tolist())

df_dcm_test = (df_dcm_filt
               [~df_dcm_filt['molecule_chembl_id'].isin(ids_usados_como_decoy)]
               .dropna(subset=['scf'])
               .drop_duplicates(subset='scf')
               .sample(min(30, len(df_dcm_filt)), random_state=99)
               .reset_index(drop=True))

df_dcm_test = df_dcm_test.rename(columns={
    'smiles':            'std_smiles',
    'molecule_chembl_id':'molecule_id',
    'scf':               'scaffold_generico',
})

print(f"✅ Muestra DCM para predicción: {len(df_dcm_test)} moléculas")
print(f"   Scaffolds únicos:            {df_dcm_test['scaffold_generico'].nunique()}")
print()
print("Estas moléculas son conocidas como INACTIVAS en múltiples targets.")
print("Si el modelo las predice como activas, puede ser:")
print("  → Un falso positivo del modelo")
print("  → Un compuesto interesante a investigar")


In [ ]:
# ── Preparar y acoplar moléculas DCM de prueba ────────────────────────────────
bitter_dock_dir = 'docking_dcm_test'
os.makedirs(bitter_dock_dir, exist_ok=True)

print(f"Preparando {len(df_dcm_test)} moléculas DCM (conformeros + PDBQT)...")
pdbqts_test = []
for _, row in tqdm(df_dcm_test.iterrows(), total=len(df_dcm_test)):
    pdbqt = preparar_ligando(row['std_smiles'], row['molecule_id'])
    pdbqts_test.append(pdbqt)
df_dcm_test['pdbqt'] = pdbqts_test
df_dcm_test = df_dcm_test[df_dcm_test['pdbqt'].notna()].reset_index(drop=True)
print(f"  Preparadas: {len(df_dcm_test)}")

# Docking
import glob as _glob
ya_proc = set(os.path.splitext(os.path.basename(f))[0]
              for f in _glob.glob(f'{bitter_dock_dir}/*.sdf'))
pendientes_t = df_dcm_test[~df_dcm_test['molecule_id'].isin(ya_proc)]

if len(pendientes_t) > 0:
    print(f"\nAcoplando {len(pendientes_t)} moléculas DCM...")
    from joblib import Parallel, delayed
    Parallel(n_jobs=min(2, os.cpu_count() or 1), prefer='threads')(
        delayed(vina_docking_decoy)(
            row.molecule_id, row.pdbqt, bitter_dock_dir
        )
        for _, row in tqdm(pendientes_t.iterrows(),
                           total=len(pendientes_t), desc='DCM test docking')
    )
sdfs_ok = len(_glob.glob(f'{bitter_dock_dir}/*.sdf'))
print(f"\n✅ SDFs disponibles: {sdfs_ok}")


In [ ]:
# ── ProLIF para moléculas DCM de prueba ──────────────────────────────────────
import glob

sdfs_dcm_test  = glob.glob(f'{bitter_dock_dir}/*.sdf')
fps_dcm_lista  = []

for sdf_path in tqdm(sdfs_dcm_test, desc='ProLIF DCM test'):
    mol_id = os.path.splitext(os.path.basename(sdf_path))[0]
    fp = get_prolif_fp(sdf_path, mol_id, protein_mol)
    if fp is not None:
        fps_dcm_lista.append(fp)

if fps_dcm_lista:
    fps_bitter_df = pd.concat(fps_dcm_lista).fillna(0)
    print(f"✅ ProLIF DCM test: {fps_bitter_df.shape[0]} moléculas × {fps_bitter_df.shape[1]} interacciones")
else:
    print("⚠️  No se calcularon fingerprints para el conjunto DCM test")
    fps_bitter_df = pd.DataFrame()


---
## 11. Estandarización y predicción final

Alineamos las columnas del fingerprint DCM test con las del training set
y aplicamos los 3 modelos. Votación mayoritaria (≥2/3 modelos).

💡 Si muchas moléculas DCM son predichas como activas, el modelo puede estar
   siendo demasiado permisivo. Si ninguna lo es, el modelo discrimina bien
   el Dark Chemical Matter — exactamente lo que queremos.


In [ ]:
# ── Estandarizar fingerprints DCM test contra el training set ───────────────
if not fps_bitter_df.empty:
    # Columnas del training set (sin 'activity')
    cols_train = [c for c in df_combined.columns
                  if c in feature_cols]

    # Reindexar: añadir columnas faltantes con 0, eliminar las extras
    X_bitter = (fps_bitter_df
                .reindex(columns=cols_train, fill_value=0)
                .values)

    n_falt = len([c for c in cols_train if c not in fps_bitter_df.columns])
    print(f"Features del training:       {len(cols_train)}")
    print(f"Features de DCM test:        {fps_bitter_df.shape[1]}")
    print(f"Features imputadas con 0:    {n_falt}")
    print(f"X_bitter shape:              {X_bitter.shape}")
else:
    print("⚠️  fps_bitter_df vacío — no se puede predecir")
    X_bitter = np.array([])


In [ ]:
# ── Predicción con los 3 modelos ─────────────────────────────────────────────
if X_bitter.size > 0:
    preds_bitter = {}
    probas_bitter = {}

    for nombre, modelo in modelos_splits.items():
        preds_bitter[nombre]  = modelo.predict(X_bitter)
        probas_bitter[nombre] = modelo.predict_proba(X_bitter)[:,1]

    df_predicciones_bitter = pd.DataFrame({
        'molecule_id':    [os.path.splitext(os.path.basename(s))[0]
                           for s in sdfs_bitter[:len(X_bitter)]],
        'P_activo_Random':      probas_bitter['Random'].round(3),
        'P_activo_Scaffold':    probas_bitter['Scaffold'].round(3),
        'P_activo_Fingerprint': probas_bitter['Fingerprint'].round(3),
        'Pred_Random':      preds_bitter['Random'],
        'Pred_Scaffold':    preds_bitter['Scaffold'],
        'Pred_Fingerprint': preds_bitter['Fingerprint'],
    })

    # Votación mayoritaria: activo si ≥ 2/3 modelos dicen 1
    df_predicciones_bitter['votos_activo'] = (
        df_predicciones_bitter[['Pred_Random','Pred_Scaffold','Pred_Fingerprint']].sum(axis=1)
    )
    df_predicciones_bitter['candidato'] = (
        df_predicciones_bitter['votos_activo'] >= 2
    )

    n_candidatos = df_predicciones_bitter['candidato'].sum()
    print(f"PREDICCIONES SOBRE BITTERDB ({len(df_predicciones_bitter)} moléculas)")
    print("=" * 60)
    print(f"  Predichos como activos (≥2/3 modelos): {n_candidatos}")
    print(f"  No activos:                            {len(df_predicciones_bitter) - n_candidatos}")
    print()
    print("TOP CANDIDATOS:")
    candidatos = df_predicciones_bitter[df_predicciones_bitter['candidato']].sort_values(
        'P_activo_Random', ascending=False)
    print(candidatos[['molecule_id','P_activo_Random','P_activo_Scaffold',
                       'P_activo_Fingerprint','votos_activo']].to_string(index=False))


In [ ]:
# ── Visualizar los top candidatos con RDKit ───────────────────────────────────
from IPython.display import display

if 'candidatos' in dir() and len(candidatos) > 0:
    # Recuperar SMILES de los candidatos
    id_a_smi = dict(zip(df_dcm_test['molecule_id'], df_dcm_test['std_smiles']))

    mols_cand, legs_cand = [], []
    for _, row in candidatos.head(12).iterrows():
        smi = id_a_smi.get(row['molecule_id'], '')
        mol = Chem.MolFromSmiles(smi)
        if mol:
            mols_cand.append(mol)
            legs_cand.append(
                f"{row['molecule_id'][:12]}\n"
                f"P={row['P_activo_Random']:.2f}/{row['P_activo_Scaffold']:.2f}"
                f"/{row['P_activo_Fingerprint']:.2f}"
            )

    if mols_cand:
        img = Draw.MolsToGridImage(
            mols_cand, molsPerRow=4, subImgSize=(300, 250),
            legends=legs_cand, returnPNG=False)
        display(img)
        img.save('results/candidatos_dcm_test.png')
        print(f"✅ Guardado: results/candidatos_dcm_test.png")
else:
    print("⚠️  No hay candidatos activos en la muestra de DCM test")


---
## 12. Guardar resultados

In [ ]:
# ── Guardar todos los resultados ─────────────────────────────────────────────
import pickle

# Dataset combinado
df_combined.to_csv('results/dataset_activos_decoys.csv', index=False)

# Métricas de los 3 splits
df_resultados_splits.to_csv('results/metricas_3splits.csv')

# Predicciones DCM test
if 'df_predicciones_bitter' in dir():
    df_predicciones_bitter.to_csv('results/predicciones_dcm_test.csv', index=False)

# Modelos
for nombre, modelo in modelos_splits.items():
    with open(f'results/xgb_{nombre.lower()}_split.pkl', 'wb') as f:
        pickle.dump(modelo, f, protocol=4)

print("✅ ARCHIVOS GUARDADOS en results/")
print("=" * 50)
for f in sorted(__import__('glob').glob('results/*')):
    tam = os.path.getsize(f) / 1024
    print(f"  {os.path.basename(f):<45} ({tam:.1f} KB)")


---
## ✅ Resumen del notebook

| Paso | Qué hace | Salida |
|------|----------|--------|
| **2. Scaffolds** | MakeScaffoldGeneric sobre activos ChEMBL | `scaffolds_activos` |
| **3. Decoys** | DCM filtrado por propiedades + scaffold distinto | `df_decoys` |
| **4. Preparación** | Conformeros 3D + PDBQT con Meeko | `pdbqt` por molécula |
| **5. Docking** | Vina batch (mismos params que NB-DOCK-02) | SDFs en `docking_decoys/` |
| **6. ProLIF** | Fingerprints de interacción (pose 1) | `fps_decoys_df` |
| **7. Dataset** | Combinar activos (1) + decoys (0) | `df_combined` |
| **8. Splits** | Random · Scaffold · Fingerprint (80/20) | 3 pares train/test |
| **9. XGBoost** | F1, MCC, Balanced Accuracy por split | `modelos_splits` |
| **10. DCM** | 30 moléculas diversas acopladas y clasificadas | `df_predicciones_dcm` |
| **11. Predicción** | Votación mayoritaria (≥2/3 modelos) | `candidatos` |

### Interpretación de los splits

- **Random** → métrica más optimista (moléculas similares en train y test)
- **Scaffold** → mide generalización a nuevas familias químicas
- **Fingerprint** → mide generalización a regiones alejadas del espacio químico

Si Random >> Scaffold ≈ Fingerprint, el modelo memoriza scaffolds del training.
Si los 3 son similares, el modelo ha aprendido patrones generalizables.
